# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² Mental Health Survey Dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. It guides you through loading the data, reviewing its structure, performing exploratory analysis, and visualizing key aspects.

### Dataset Source
The FAIR² dataset's Croissant schema is accessible at:
- https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}, Version: {metadata.version}\n")
print(f"Keywords: {', '.join(metadata.keywords)}")
print(f"License: {metadata.license}")

## 2. Data Overview
Examine available record sets, the fields within each, and their `@id`s. This helps identify what data can be extracted for analysis. All references use their Croissant `@id` identifiers to ensure traceability.

**List available record sets:**

In [ ]:
# List the available record sets and their field IDs
print("Record Sets available in dataset:")
for rs in dataset.record_sets:
    recset_id = rs['@id']
    recset_name = rs.get('name', '')
    print(f"  - RecordSet @id: {recset_id}\n    Name: {recset_name}")
    # List fields and columns
    field_ids = rs.get('field', [])
    if isinstance(field_ids, dict):
        field_ids = [field_ids]
    for field_id in field_ids:
        if isinstance(field_id, dict):
            fid = field_id.get('@id', str(field_id))
        else:
            fid = field_id
        print(f"    - Field @id: {fid}")
    col_ids = rs.get('column', [])
    for col_id in col_ids:
        if isinstance(col_id, dict):
            cid = col_id.get('@id', str(col_id))
        else:
            cid = col_id
        print(f"    - Column @id: {cid}")
    print()

**Preview a sample record from each record set (`@id`):**

In [ ]:
# List a sample record for each record set:
for recset in dataset.record_sets:
    recset_id = recset['@id']
    print(f"Sample from record set: {recset_id}")
    try:
        for i, rec in enumerate(dataset.records(record_set=recset_id)):
            print(rec)
            break  # Only print the first record
    except Exception as e:
        print(f"Could not retrieve records for {recset_id}: {e}")
    print()

## 3. Data Extraction
Load data from each record set into a pandas DataFrame with column headers as the dataset `@id` fields.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record set @ids to extract:", record_set_ids)

dataframes = dict()
for recset_id in record_set_ids:
    print(f"Loading records for {recset_id} ...")
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    if not df.empty:
        print(df.head(2))
    print()

# For this dataset, select the principal survey record set for main EDA. Replace with the actual @id you want to analyze:
main_record_set_id = record_set_ids[0]  # Choose the primary record set
df_main = dataframes[main_record_set_id]
print(f"Using main record set @id for EDA: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Let's explore and process the main survey DataFrame. We'll perform:
- Filtering of a key numeric field from the survey (for example, PHQ-9 score or age), using `@id` as the column.
- Normalization of that field.
- Grouping by demographic fields (e.g., by gender or education level, referenced by their `@id`).

In [ ]:
# Choose a numeric field and group field by @id for EDA
# Replace with precise field @ids (use the output of previous cells to list available fields)
numeric_field_id = None
group_field_id = None

# Try to heuristically guess the PHQ-9 or GAD-7 or age field from columns:
for col in df_main.columns:
    if 'phq' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    for col in df_main.columns:
        if 'gad' in col.lower():
            numeric_field_id = col
            break
if not numeric_field_id:
    for col in df_main.columns:
        if 'age' in col.lower():
            numeric_field_id = col
            break
if not numeric_field_id:
    numeric_field_id = df_main.select_dtypes(include='number').columns[0]  # fallback

# Heuristically pick a group field:
for col in df_main.columns:
    if 'gender' in col.lower():
        group_field_id = col
        break
if not group_field_id:
    for col in df_main.columns:
        if 'education' in col.lower():
            group_field_id = col
            break
if group_field_id is None and len(df_main.columns) >= 2:
    group_field_id = df_main.columns[1]  # arbitrarily take the second column

print(f"Numeric field selected: {numeric_field_id}")
print(f"Group field selected: {group_field_id}")

# Filter out records with non-numeric or missing values
filtered_df = df_main.copy()
filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').notnull()]
filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')

threshold = filtered_df[numeric_field_id].mean()  # set threshold as mean for demo
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} field:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if present
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
    print(f"\nGrouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field (e.g., PHQ-9, GAD-7, or Age) and compare across categories in the selected demographic field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], bins=15, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f'Distribution of {numeric_field_id}')
plt.show()

# Boxplot by group (if group field is categorical)
if group_field_id in filtered_df.columns and filtered_df[group_field_id].nunique() <= 10:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² Mental Health Survey Dataset using the Croissant metadata standard and `mlcroissant` library. Analysis leveraged the dataset's semantic structure (via `@id`s), with typical filtering, normalization, and visualization steps applicable to survey data. For more extensive analysis, refer to the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) and the published FAIR² dataset metadata.
